# Structure output of API response

In [1]:
from pydantic import BaseModel
from google import genai

## Simple example

In [2]:
CLIENT = genai.Client()
MODEL_NAME = "gemini-3.1-flash-lite-preview"

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


In [3]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [ ]:
response = CLIENT.models.generate_content(
    model=MODEL_NAME,
    contents=[
        {"role": "system", "parts": [{"text": "Extract the event information."}]},
        {
            "role": "user",
            "parts": [{"text": "Alice and Bob are going to a science fair on Friday."}],
        },
    ],
    config={
        "response_mime_type": "application/json",
        "response_json_schema": CalendarEvent.model_json_schema(),
    },
)

In [10]:
print(response.text)

{
  "name": "science fair",
  "date": "Friday",
  "participants": [
    "Alice",
    "Bob"
  ]
}


In [11]:
event = CalendarEvent.model_validate_json(response.text)
print(event)

name='science fair' date='Friday' participants=['Alice', 'Bob']


## Structured documentation RAG

In [13]:
import json

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")


Indexed 385 chunks from 95 documents


In [20]:
def search(query):
    return index.search(query, num_results=5)


instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )


def llm(user_prompt, instructions=None, model=MODEL_NAME):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "parts": [{"text": instructions}]
        })

    messages.append({
        "role": "user", 
        "parts": [{"text": user_prompt}]
    })

    response = CLIENT.models.generate_content(
        model=model,
        contents=messages
    )

    return response.text


def rag(query):
    
    instructions = """
    You're a course assistant, your task is to answer the QUESTION from the
    course students using the provided CONTEXT
    """
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [21]:
answer = rag('how do I implement LLM as a judge?')
print(answer)

To implement an LLM as a judge using the Evidently library, you should follow these steps:

### 1. Installation and Setup
First, install the library and set your OpenAI API key:
```python
pip install evidently
```
```python
import os
os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
```

### 2. Create Your Dataset
Prepare a dataset (as a pandas DataFrame) containing the data you want to evaluate, such as:
* **Questions**
* **New responses** (the model outputs you are testing)
* **Target responses** (if using a reference-based approach)

### 3. Design the Judge (Define the Prompt Template)
You can define custom criteria for your judge using prompt templates. For example, to create a verbosity judge:
```python
from evidently.llm.templates import BinaryClassificationPromptTemplate
from evidently.descriptors import LLMEval

verbosity = BinaryClassificationPromptTemplate(
    template="""You are an expert text evaluator. Evaluate if the response is concise or verbose.
    ... [your specific instruc

### Structured RAG

In [30]:
def llm_structured(
    user_prompt, 
    output_type,
    instructions=None, 
    model=MODEL_NAME,
):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "parts": [{"text": instructions}]
        })

    messages.append({
        "role": "user", 
        "parts": [{"text": user_prompt}]
    })

    response = CLIENT.models.generate_content(
        model=model,
        contents=messages,
        config={
            "response_mime_type": "application/json",
            "response_json_schema": output_type.model_json_schema(),
        },
    )

    return output_type.model_validate_json(response.text)


In [47]:
from typing import Literal, Optional
from pydantic import Field

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [42]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [ ]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [46]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)
print(answer.found_answer)

answer='The provided documentation does not contain information about how to install Kafka on Windows.' found_answer=False confidence=1.0 confidence_explanation='The documentation provided is exclusively about the Evidently platform (installation, evaluation workflows, and synthetic data), and contains no references to Apache Kafka.' answer_type='reference' followup_questions=['How do I install the Evidently Python library?', 'What are the supported installation methods for Evidently?']
The provided documentation does not contain information about how to install Kafka on Windows.
1.0
The documentation provided is exclusively about the Evidently platform (installation, evaluation workflows, and synthetic data), and contains no references to Apache Kafka.
reference
['How do I install the Evidently Python library?', 'What are the supported installation methods for Evidently?']
False


In [49]:
from pydantic import model_validator

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [50]:
AnswerResponse.model_json_schema()


{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [54]:
from pydantic import ValidationError

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

def rag_structured(query, output_type=AnswerResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    try:
        return llm_structured(
            instructions=instructions,
            user_prompt=prompt,
            output_type=output_type,
        )
    except ValidationError as e:
        print(f"Validation error: {e}")
        return None

In [52]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided documentation contains information about installing and using the Evidently library, but it does not contain instructions on how to install Apache Kafka.'), found_answer=False, answer=None)

In [55]:
answer = rag_structured('how do I run llm evals??', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='To run LLM evaluations using the Evidently library, follow these steps:\n\n1. **Prepare your data**: Create an Evidently `Dataset` object from your data. You must compute and add `descriptors` to this dataset first.\n2. **Configure the Report**: Use the `Report` class with `TextEvals()` as a metric.\n   ```python\n   report = Report(metrics=[TextEvals()])\n   ```\n3. **Run the evaluation**: Execute the report using the `run()` method, passing your dataset(s).\n   ```python\n   my_eval = report.run(eval_data, None)\n   ```\n4. **Explore results**: You can view the report directly in an interactive Python environment (like Jupyter) or upload the results to an Evidently workspace using `ws.add_run(project.id, my_eval)`.\n\nFor more advanced setups, you can add pass/fail `Tests` conditions or run comparisons between two datasets by passing both to the `run()` method.', found_answer=True, confidence=1.0, con